# Searching MAUDE Data

Part of the [PyMAUDE](../) examples — see [`quickstart.ipynb`](../quickstart.ipynb) first if you haven't loaded a database yet.

This notebook covers the ways to find events: exact-field queries, substring search (with OR/AND/grouped logic), and event narratives.

---
## Contents
1. [Setup](#1-setup)
2. [Exact-field queries](#2-exact)
3. [Substring search](#3-search)
4. [Grouped search across device classes](#4-grouped)
5. [Event narratives](#5-narratives)

---
## 1. Setup <a id="1-setup"></a>

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = '../maude.duckdb'
DATA_DIR = '../maude_data'
YEARS    = '2024-2026'

db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')
db.add_years(YEARS, tables=['master', 'device', 'text'], download=False)

In [ ]:
db.info()

---
## 2. Exact-field queries <a id="2-exact"></a>

`query_device()` does **exact** matching: case-insensitive for `brand_name`, `generic_name`, and `manufacturer_name`, but **case-sensitive** for `product_code` (FDA product codes are always uppercase, e.g. `'NIQ'` — `'niq'` won't match). Combine any of these plus `start_date`/`end_date`.

In [ ]:
# All events for a specific product code (NIQ = venous stents)
niq = db.query_device(product_code='NIQ')
print(f'NIQ (venous stent) events loaded: {len(niq):,}')
niq[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']].head(10)

In [ ]:
# Narrow to a specific manufacturer + date window
# Note: manufacturer_name is an EXACT match (see the docstring above), and
# MANUFACTURER_D_NAME in the raw data includes the legal entity suffix —
# 'Boston Scientific' alone won't match 'BOSTON SCIENTIFIC CORPORATION'.
bsci_jan_thru_aug_2024 = db.query_device(
    manufacturer_name='Boston Scientific Corporation',
    start_date='2024-01-01',
    end_date='2024-08-31'
)

print(f'Boston Scientific Corporation events in 2024 Jan-Aug: {len(bsci_jan_thru_aug_2024):,}')
bsci_jan_thru_aug_2024[['MDR_REPORT_KEY', 'BRAND_NAME', 'EVENT_TYPE', 'MANUFACTURER_G1_NAME', 'DATE_RECEIVED']]

---
## 3. Substring search <a id="3-search"></a>

`search_by_device_names()` does **case-insensitive substring matching** across a synthesized `DEVICE_NAME_CONCAT` column (BRAND_NAME | GENERIC_NAME | MANUFACTURER_D_NAME concatenated). This is useful because MAUDE entries are often inconsistent, with names of medical devices often appearing in one of these three columns.

| Criteria format | Logic |
|---|---|
| `'term'` | single substring |
| `['a', 'b']` | a **OR** b |
| `[['a', 'b'], 'c']` | (a **AND** b) **OR** c |
| `{'group1': ..., 'group2': ...}` | grouped (see [section 4](#4-grouped)) |

In [ ]:
# Simple substring — anything with "venous stent" in any name field
venous_stents = db.search_by_device_names('venous stent')
print(f'Events matching "venous stent": {len(venous_stents):,}')
venous_stents[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'DATE_RECEIVED']]

In [ ]:
# OR logic — venous stent OR iliac stent
venous_iliac = db.search_by_device_names(['venous stent', 'iliac stent'])
print(f'Venous OR iliac stents: {len(venous_iliac):,}')
venous_iliac['GENERIC_NAME'].value_counts().head(10)

In [ ]:
# AND logic — must contain both "argon" AND "cleaner" (avoids false positives)
argon_cleaner = db.search_by_device_names([['argon','cleaner']])
print(f'Argon AND Cleaner events: {len(argon_cleaner):,}')
argon_cleaner[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']]

In [ ]:
# Combined: (argon AND cleaner) OR (angiojet) OR (thrombectomy)
thrombectomy_broad = db.search_by_device_names(
    [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy']
)
print(f'Broad rotational thrombectomy search: {len(thrombectomy_broad):,}')
thrombectomy_broad[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME']].head(10)

Results are pandas DataFrames — export anytime:

In [ ]:
thrombectomy_broad.to_csv('thrombectomy_broad.csv', index=False)

---
## 4. Grouped search across device classes <a id="4-grouped"></a>

A **dict** of criteria runs multiple searches at once and labels each result with a `search_group` column. Useful for comparative studies.

In [ ]:
grouped = db.search_by_device_names({
    'venous_stents':  ['venous stent', 'venous stenting'],
    'thrombectomy':   [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'],
})

print(f'Total events: {len(grouped):,}')
print('\nEvents by group:')
print(grouped['search_group'].value_counts().to_string())

In [ ]:
grouped[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'search_group']].head(15)

---
## 5. Event narratives <a id="5-narratives"></a>

`get_narratives()` fetches the free-text FOI_TEXT descriptions for a set of MDR_REPORT_KEYs.

In [ ]:
narratives = db.get_narratives(thrombectomy_broad['MDR_REPORT_KEY'])
print(f'Narratives retrieved: {len(narratives):,}')
narratives.head(5)

In [ ]:
# Read a few narratives in full
for _, row in narratives.head(3).iterrows():
    print(f"=== MDR {row['MDR_REPORT_KEY']} ===")
    print(row['FOI_TEXT'][:500])
    print()

In [ ]:
db.close()